<a href="https://colab.research.google.com/github/Ali-Alameer/Deep-Learning/blob/main/week7_object_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train YOLO11 on a Custom Dataset

### **Steps Covered in this Tutorial**

To train our detector we take the following steps:

* Install YOLO11 dependencies
* Load custom dataset from Roboflow in YOLO11 format
* Run YOLO11 training
* Evaluate YOLO11 performance
* Run YOLO11 inference on test images
* OPTIONAL: Deployment
* OPTIONAL: Active Learning


### Preparing a Custom Dataset

If you already have your own images (and, optionally, annotations), you can convert your dataset using [Roboflow](https://roboflow.com), a set of tools developers use to build better computer vision models quickly and accurately. 1

#Install Dependencies

_(Remember to choose GPU in Runtime if not already selected. Runtime --> Change Runtime Type --> Hardware accelerator --> GPU)_

In [1]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


## Preparing a custom dataset

### Step 1: Creating project

Before you start, you need to create a Roboflow [account](https://app.roboflow.com/login). Once you do that, you can create a new project in the Roboflow [dashboard](https://app.roboflow.com/). Keep in mind to choose the right project type. In our case, Object Detection.

<div align="center">
  <img
    width="640"
    src="https://ik.imagekit.io/roboflow/preparing-custom-dataset-example/creating-project.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672929799852"
  >
</div>

### Step 2: Uploading images

Next, add the data to your newly created project. You can do it via API or through our [web interface](https://docs.roboflow.com/adding-data/object-detection).

If you drag and drop a directory with a dataset in a supported format, the Roboflow dashboard will automatically read the images and annotations together.

<div align="center">
  <img
    width="640"
    src="https://ik.imagekit.io/roboflow/preparing-custom-dataset-example/uploading-images.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672929808290"
  >
</div>

### Step 3: Labeling

If you only have images, you can label them in [Roboflow Annotate](https://docs.roboflow.com/annotate).

<div align="center">
  <img
    width="640"
    src="https://user-images.githubusercontent.com/26109316/210901980-04861efd-dfc0-4a01-9373-13a36b5e1df4.gif"
  >
</div>

### Step 4: Generate new dataset version

Now that we have our images and annotations added, we can Generate a Dataset Version. When Generating a Version, you may elect to add preprocessing and augmentations. This step is completely optional, however, it can allow you to significantly improve the robustness of your model.

<div align="center">
  <img
    width="640"
    src="https://media.roboflow.com/preparing-custom-dataset-example/generate-new-version.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1673003597834"
  >
</div>

### Step 5: Exporting dataset

Once the dataset version is generated, we have a hosted dataset we can load directly into our notebook for easy training. Click `Export` and select the `desired` dataset format.

<div align="center">
  <img
    width="640"
    src="https://ik.imagekit.io/roboflow/preparing-custom-dataset-example/export.gif?ik-sdk-version=javascript-1.4.3&updatedAt=1672943313709"
  >
</div>




In [2]:
!pip install roboflow ultralytics

  Using cached idna-3.7-py3-none-any.whl.metadata (9.9 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
Using cached idna-3.7-py3-none-any.whl (66 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 MB 17.9 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.3/892.3 kB 11.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 9.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 15.8 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 14.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 18.3 MB/s  0:00:04m0:00:0100:01
Using cached networkx-3.5-py3-none-any.whl (2.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3

# Download Correctly Formatted Custom Data

Next, we'll download our dataset in the right format. Use the `YOLO11 PyTorch` export. Note that this model requires YOLO TXT annotations, a custom YAML file, and organized directories. The roboflow export writes this for us and saves it in the correct spot.


In [3]:
!curl -L "https://app.roboflow.com/ds/PEtxKedfAz?key=kVGmcyjqOg" -o roboflow.zip
!unzip roboflow.zip -d roboflow


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   904  100   904    0     0   3027      0 --:--:-- --:--:-- --:--:--  3023- --:--:-- --:--:--     0
100  691M  100  691M    0     0  18.7M      0  0:00:36  0:00:36 --:--:-- 21.5M1.3M      0  0:01:01  0:00:07  0:00:54 13.2M 0  15.1M      0  0:00:45  0:00:16  0:00:29 17.9M44  0:00:17  0:00:27 18.6M 0  16.7M      0  0:00:41  0:00:23  0:00:18 20.7M 0:00:39  0:00:27  0:00:12 21.7M
Archive:  roboflow.zip
  inflating: roboflow/README.dataset.txt  
  inflating: roboflow/README.roboflow.txt  
  inflating: roboflow/data.yaml      
   creating: roboflow/test
   creating: roboflow/test/images
 extracting: roboflow/test/images/000003_jpg.rf.04ce6f58e661d17da480fcaf367b2fcd.jpg  
 extracting: roboflow/test/images/000013_jpg.rf.ac1b6949f31fb1baebdc0ebc068a5bea.jpg  
 extracting: roboflow/test/images/000019_jpg.rf.a7b52470a9e73c6202a9b4b96909

# Begin Custom Training

We're ready to start custom training.

NOTE: We will only modify one of the YOLO11 training defaults in our example: `epochs`. We will adjust from 30 to 10 epochs in our example for speed.

In [4]:
# run this cell to begin training; for real implemenation increase number of epochs
!yolo detect train model=yolo11n.pt data=roboflow/data.yaml epochs=5 imgsz=640


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/chidimo/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.222 🚀 Python-3.13.5 torch-2.9.0 CPU (Apple M1 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=roboflow/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=

In [5]:
from IPython.display import Image, display

display(Image(filename='/content/runs/detect/train/results.png'))

FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/train/results.png'

# Evaluation

We can evaluate the performance of our custom training using the provided evalution script. Make sure the exp #number is up to date -- check the folder within

In [ ]:
# Run evaluation
!yolo predict \
  model=/content/runs/detect/train/weights/best.pt \
  source=/content/roboflow/test/images \
  project=/content/runs/detect \
  name=pred_test \
  exist_ok=True


In [ ]:
import gradio as gr
import glob
from PIL import Image

# folder where YOLO saves predictions
PRED_DIR = "/content/runs/detect/pred_test"

# fetch list of image paths (JPG and PNG support)
images = sorted(glob.glob(f"{PRED_DIR}/*.jpg")) + sorted(glob.glob(f"{PRED_DIR}/*.png"))

def show_image(image_path):
    return Image.open(image_path)

with gr.Blocks() as demo:
    gr.Markdown("## 🐄 YOLO Detection Results Viewer")

    dropdown = gr.Dropdown(
        choices=images,
        label="Select a prediction image",
        value=images[0] if images else None
    )

    image_display = gr.Image(label="Detected Image")

    dropdown.change(fn=show_image, inputs=dropdown, outputs=image_display)

demo.launch(debug=True)
